# 03 – Naive Bayes
**CS 5331/7331 · Project Team Five**

Trains Gaussian, Bernoulli and Categorical NB on both feature sets (full 30 / no-external 25), with and without SMOTEN.
Hyperparameters are tuned with 5-fold stratified CV on the training set (scored on F1), then each best model is scored once on the test set.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q imbalanced-learn

import os, json
import pandas as pd
from sklearn.naive_bayes import GaussianNB, BernoulliNB, CategoricalNB
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTEN

BASE = '/content/drive/MyDrive/CS7331_Phishing'
DATA_DIR = f'{BASE}/data'
RESULTS_DIR = f'{BASE}/results'  # new folder for model scores and predictions
os.makedirs(RESULTS_DIR, exist_ok=True)
SEED = 42

# load the exact splits from preprocessing
X_train = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_test = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')['phishing']
y_test = pd.read_csv(f'{DATA_DIR}/y_test.csv')['phishing']
with open(f'{DATA_DIR}/feature_sets.json') as f:
    FEATURE_SETS = json.load(f)

print(X_train.shape, X_test.shape, y_train.mean().round(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(4679, 30) (1170, 30) 0.516


In [9]:
from joblib import Memory
CACHE = Memory('/content/smote_cache', verbose=0)  # remembers SMOTEN results so it doesn't redo the same thing
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def shift(X):
    return X + 1  # CategoricalNB needs categories as 0,1,2 instead of -1,0,1

MODELS = {
    # Gaussian is in the original plan, kept as a baseline even though our features aren't continuous
    'GaussianNB': (GaussianNB(), {'nb__var_smoothing': [1e-9, 1e-5, 1e-3, 1e-2, 0.1, 0.5, 1.0, 5.0]}),
    # Bernoulli treats each feature as on/off (>0 = 1), so 0 "suspicious" gets lumped with -1
    'BernoulliNB': (BernoulliNB(), {'nb__alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]}),
    # Categorical keeps all three levels, best fit for this data on paper
    'CategoricalNB': (CategoricalNB(min_categories=3), {'nb__alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]}),
}

def make_pipe(name, model, use_smote):
    steps = []
    if use_smote:
        steps.append(('smote', SMOTEN(random_state=SEED)))  # inside the pipeline so it only touches training folds
    if name == 'CategoricalNB':
        steps.append(('shift', FunctionTransformer(shift)))
    steps.append(('nb', model))
    return Pipeline(steps, memory=CACHE)

In [10]:
rows = []
preds = pd.DataFrame({'y_true': y_test.values})

for fs_name, cols in FEATURE_SETS.items():
    for use_smote in [False, True]:
        for name, (model, grid) in MODELS.items():
            search = GridSearchCV(make_pipe(name, model, use_smote), grid, scoring='f1', cv=cv)
            search.fit(X_train[cols], y_train)
            print(f"done: {name} | {fs_name} | {'smote' if use_smote else 'no_smote'}")

            best = search.best_estimator_  # refit on the full training set with the best setting
            y_pred = best.predict(X_test[cols])
            y_prob = best.predict_proba(X_test[cols])[:, 1]

            smote_tag = 'smote' if use_smote else 'no_smote'
            rows.append({
                'model': name, 'features': fs_name, 'smote': smote_tag,
                'best_params': search.best_params_,
                'cv_f1': round(search.best_score_, 4),
                'accuracy': round(accuracy_score(y_test, y_pred), 4),
                'precision': round(precision_score(y_test, y_pred), 4),
                'recall': round(recall_score(y_test, y_pred), 4),
                'f1': round(f1_score(y_test, y_pred), 4),
                'roc_auc': round(roc_auc_score(y_test, y_prob), 4),
            })
            # keep predictions for the confusion matrices and ROC curves later
            key = f'{name}|{fs_name}|{smote_tag}'
            preds[f'{key}|pred'] = y_pred
            preds[f'{key}|prob'] = y_prob

results = pd.DataFrame(rows)
results.sort_values('cv_f1', ascending=False)

done: GaussianNB | full | no_smote
done: BernoulliNB | full | no_smote
done: CategoricalNB | full | no_smote
done: GaussianNB | full | smote
done: BernoulliNB | full | smote
done: CategoricalNB | full | smote
done: GaussianNB | no_external | no_smote
done: BernoulliNB | no_external | no_smote
done: CategoricalNB | no_external | no_smote
done: GaussianNB | no_external | smote
done: BernoulliNB | no_external | smote
done: CategoricalNB | no_external | smote


,model,features,smote,best_params,cv_f1,accuracy,precision,recall,f1,roc_auc
2,CategoricalNB,full,no_smote,{'nb__alpha': 5.0},0.9305,0.9256,0.9434,0.9106,0.9267,0.9798
5,CategoricalNB,full,smote,{'nb__alpha': 0.01},0.9296,0.9248,0.9448,0.9073,0.9257,0.9802
11,CategoricalNB,no_external,smote,{'nb__alpha': 5.0},0.9268,0.9197,0.9427,0.8990,0.9203,0.9750
8,CategoricalNB,no_external,no_smote,{'nb__alpha': 5.0},0.9266,0.9188,0.9380,0.9023,0.9198,0.9751
3,GaussianNB,full,smote,{'nb__var_smoothing': 0.1},0.9054,0.8923,0.8723,0.9272,0.8989,0.9635
1,BernoulliNB,full,no_smote,{'nb__alpha': 2.0},0.9036,0.8889,0.8898,0.8957,0.8927,0.9577
4,BernoulliNB,full,smote,{'nb__alpha': 0.01},0.9026,0.8889,0.8924,0.8924,0.8924,0.9582
0,GaussianNB,full,no_smote,{'nb__var_smoothing': 0.1},0.9023,0.8897,0.8671,0.9288,0.8969,0.9632
9,GaussianNB,no_external,smote,{'nb__var_smoothing': 0.1},0.8996,0.8915,0.8709,0.9272,0.8982,0.9587
6,GaussianNB,no_external,no_smote,{'nb__var_smoothing': 0.1},0.8971,0.8923,0.8688,0.9321,0.8994,0.9589


In [13]:
results.to_csv(f'{RESULTS_DIR}/nb_results.csv', index=False)
preds.to_csv(f'{RESULTS_DIR}/nb_predictions.csv', index=False)
print("Saved nb_results.csv and nb_predictions.csv to /results")

Saved nb_results.csv and nb_predictions.csv to /results
